In [12]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
# from sklearn.model_selection import train_test_split

# # Example dataset (Iris)
# from sklearn.datasets import load_iris
# data = load_iris()
# X, y = data.data, data.target

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# # Train Random Forest on TEST dataset
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# rf.fit(X_test, y_test)

# # Step 1: Get leaf memberships for test (train set for RF) and train (query set)
# test_leaves = rf.apply(X_test)   # shape: (n_test, n_trees)
# train_leaves = rf.apply(X_train) # shape: (n_train, n_trees)

# # Step 2: Build mapping from leaf -> train indices
# leaf_to_train = {}
# for i, leaves in enumerate(train_leaves):
#     for tree_idx, leaf_id in enumerate(leaves):
#         leaf_to_train.setdefault((tree_idx, leaf_id), []).append(i)

# # Step 3: For each test instance, find neighbors from train dataset
# neighbors = []
# for test_idx, leaves in enumerate(test_leaves):
#     neigh_set = set()
#     for tree_idx, leaf_id in enumerate(leaves):
#         neigh_set.update(leaf_to_train.get((tree_idx, leaf_id), []))
#     neighbors.append(list(neigh_set))

# # Example: print neighbors for first 5 test instances
# for i in range(5):
#     print(f"Test instance {i} has {len(neighbors[i])} neighbors in train set: {neighbors[i][:10]} ...")


In [15]:
def calculate_neighbors(X_test, y_test, X_train):

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=2500, random_state=42)
    rf.fit(X_test, y_test)

    # Step 1: Get leaf memberships for test (train set for RF) and train (query set)
    test_leaves = rf.apply(X_test)   # shape: (n_test, n_trees)
    train_leaves = rf.apply(X_train) # shape: (n_train, n_trees)

    # Step 2: Build mapping from leaf -> train indices
    leaf_to_train = {}
    for i, leaves in enumerate(train_leaves):
        for tree_idx, leaf_id in enumerate(leaves):
            leaf_to_train.setdefault((tree_idx, leaf_id), []).append(i)

    # Step 3: For each test instance, find neighbors from train dataset
    neighbors = []
    for test_idx, leaves in enumerate(test_leaves):
        neigh_set = set()
        for tree_idx, leaf_id in enumerate(leaves):
            neigh_set.update(leaf_to_train.get((tree_idx, leaf_id), []))
        neighbors.append(list(neigh_set))
    
    return neighbors

In [ ]:
directory = "model_preds"
list_of_datasets = os.listdir(directory)

df_all = pd.DataFrame()

for dataset in tqdm(list_of_datasets):
    for model in ['lin', 'xgb', 'svm']:
        
        test_path = os.path.join(directory, dataset, model, "test_org.csv")
        train_path = os.path.join(directory, dataset, model, "train_org.csv")
        if os.path.exists(test_path) and os.path.exists(train_path):
            test = pd.read_csv(test_path)
            train = pd.read_csv(train_path)

            X_test = test.drop(columns=["name","is_train","target","prediction"])
            X_test = X_test.drop(columns=X_test.columns[X_test.columns.str.startswith('score_')].to_list())
            y_test = test['prediction']

            X_train = train.drop(columns=["name","is_train","target","prediction"])
            X_train = X_train.drop(columns=X_train.columns[X_train.columns.str.startswith('score_')].to_list())
            y_train = train['prediction']

            neighbors = calculate_neighbors(X_test, y_test.values, X_train)
            neighborhood_size = [len(n) for n in neighbors]
            neighborhood_size_pct = np.array(neighborhood_size) / len(X_train) * 100

            df = pd.DataFrame({
                "neighborhood_size": neighborhood_size,
                "neighborhood_size_pct": neighborhood_size_pct
            })
            df["dataset"] = dataset
            df["model"] = model

            df_all = pd.concat([df_all, df], ignore_index=True)

df_all.to_csv("results/rf_neighborhood_sizes_2500.csv", index=False)
# 2:06 dla 100
# 1:15:53 dla 2500           

100%|██████████| 18/18 [1:15:53<00:00, 252.99s/it]  
